# 6. Beliefs MVP

This notebook walks through the legacy MVP belief-updating protocol step by step, from canonical cache initialization to the final four-rung experiment.


## Reading Guide

This notebook is intentionally verbose and sequential. The goal is to make the data flow easy to audit:

1. load the canonical cache;
2. inspect market and probability tables;
3. build or load the shared belief manifest;
4. trace one concrete example from target market to sibling context;
5. inspect the tensorized sample seen by the DeepSets encoder;
6. define the split, features, models, losses, and metrics explicitly;
7. run the MVP ladder and inspect the results.


## MVP Protocol

The MVP asks a proof-of-concept question: can fresh non-local family context improve terminal-outcome prediction when the target market is only observed through a stale local view?

Protocol summary:

1. target market `A` is observed locally only at stale time `t-Δ`;
2. siblings from the same weak family are observed at context time `t`;
3. the target market's own current state at `t` is hidden;
4. the label is the terminal binary outcome;
5. the ladder compares stale-only, stale-plus-raw, stale-plus-embedding, and corrupted-context controls.


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import torch
import torch.nn as nn
from IPython.display import display
from sklearn.impute import SimpleImputer
from sklearn.metrics import log_loss, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import HistGradientBoostingClassifier

from polymarket_research.belief_updating import (
    AGG_CONTEXT_FEATURE_NAMES,
    CONTEXT_FEATURE_NAMES,
    STALE_FEATURE_NAMES,
    BeliefUpdatingDatasetBuilder,
    BeliefUpdatingManifest,
    BeliefUpdatingSpec,
    BeliefUpdatingTorchDataset,
    DeepSetsEncoder,
    BeliefUpdatingPredictor,
    BeliefUpdatingMVPExperiment,
    BeliefUpdatingMVPConfig,
)
from polymarket_research.data.canonical import CanonicalDataset
from polymarket_research.utils import setup_root


## Step 1. Initialize Paths and Load the Canonical Dataset


In [ ]:
REPO_ROOT = setup_root()
DATA_SOURCE = 'polymarket'  # or 'kalshi'
ARTEFACT_ROOT = REPO_ROOT / 'frozen_notebooks' / 'running_artefacts' / DATA_SOURCE
CANONICAL_CACHE_DIR = ARTEFACT_ROOT / 'canonical_dataset'
MANIFEST_CACHE_DIR = ARTEFACT_ROOT / 'belief_updating_manifest'

canonical = CanonicalDataset.from_parquet(CANONICAL_CACHE_DIR)
spec = BeliefUpdatingSpec(
    stale_horizon_hours=24,
    context_lookback_hours=72,
    target_horizon_hours=24,
    family_context_limit=16,
    time_split_quantile=0.8,
)
protocol = BeliefUpdatingDatasetBuilder(canonical, spec).build()


In [ ]:
display(canonical.summary())
if canonical.download_status is not None:
    display(canonical.status().head())


## Step 2. Inspect the Canonical Market Table

The market table gives us the target universe, timing metadata, text fields, and weak family assignments.


In [ ]:
display(canonical.markets[[
    'market_id', 'question', 'research_category', 'family_id', 'created_at', 'end_date', 'final_yes_probability', 'probability_rows'
]].head(10))

family_sizes = canonical.markets.groupby('family_id').size().reset_index(name='family_size').sort_values('family_size', ascending=False)
display(family_sizes.head(10))


## Step 3. Inspect the Canonical Probability Panel

This is the time-indexed source used to build stale-local features and fresh sibling context snapshots.


In [ ]:
display(canonical.probabilities.head(10))

probability_coverage = canonical.probabilities.groupby("market_id").agg(
    rows=("timestamp_utc", "size"),
    start=("timestamp_utc", "min"),
    end=("timestamp_utc", "max"),
).reset_index()
display(probability_coverage.head(10))


## Step 4. Build or Load the Shared Belief Manifest

The shared manifest is the key bridge between canonical data and both belief-updating protocols. It stores stale-local features, per-sibling context snapshots, aggregated context features, and all labels needed by both notebooks.


In [ ]:
display(manifest.summary())
print("stale feature count:", len(manifest.stale_feature_names))
print("context feature count:", len(manifest.context_feature_names))
print("global feature count:", len(manifest.global_feature_names))


## Step 5. Inspect the Example Table

Each example is one `(target market, context time, stale time, horizon, delta)` row.


In [ ]:
examples = manifest.examples.copy()
display(
    examples[[
        "market_id", "family_id", "horizon_hours", "delta_hours_int", "context_time", "stale_time",
        "n_siblings", "stale_yes_probability", "label", "label_update_logit", "target_current_probability"
    ]].head(10)
)

display(
    examples.groupby(["horizon_hours", "delta_hours_int"]).agg(
        n_examples=("label", "size"),
        pct_yes=("label", "mean"),
        mean_siblings=("n_siblings", "mean"),
    ).reset_index()
)


## Step 6. Trace One Example End to End

We choose one rich example and inspect the target trajectory around stale time and context time.


In [ ]:
trace_row = examples.sort_values(["n_siblings", "delta_hours_int", "horizon_hours"], ascending=[False, False, True]).iloc[0]
trace_market_id = str(trace_row["market_id"])
trace_context_time = pd.Timestamp(trace_row["context_time"])
trace_stale_time = pd.Timestamp(trace_row["stale_time"])
trace_family_id = str(trace_row["family_id"])

display(trace_row.to_frame(name="value").head(25))

target_panel = canonical.probabilities.loc[canonical.probabilities["market_id"].astype(str) == trace_market_id].copy()
target_window = target_panel.loc[
    target_panel["timestamp_utc"].between(trace_stale_time - pd.Timedelta(hours=36), trace_context_time + pd.Timedelta(hours=1))
].copy()
display(target_window.tail(20))


## Step 7. Inspect the Sibling Context Used at Context Time

The manifest stores sibling snapshots separately so the set encoder can rebuild variable-length context sets on demand.


In [ ]:
context_rows = manifest.context_snapshots.loc[
    (manifest.context_snapshots["family_id"].astype(str) == trace_family_id)
    & (manifest.context_snapshots["snapshot_time"] == trace_context_time)
    & (manifest.context_snapshots["market_id"].astype(str) != trace_market_id)
].copy()
context_rows["ctx_prob_vs_target_stale"] = context_rows["ctx_yes_probability"] - float(trace_row["stale_yes_probability"])
display(context_rows[[
    "market_id", "snapshot_time", "ctx_yes_probability", "ctx_confidence_margin", "ctx_prob_vs_target_stale",
    "ctx_lookback_24h_change", "ctx_lookback_24h_volatility", "ctx_lookback_24h_log_trade"
]].head(20))


## Step 8. Inspect the Tensorized Sample Seen by the DeepSets Encoder

This is the exact format used by the PyTorch rungs: one flat stale vector, one padded context matrix, and a boolean mask.


In [ ]:
torch_dataset = BeliefUpdatingTorchDataset(manifest, indices=[int(trace_row.name)], max_context_size=16)
sample = torch_dataset[0]
print("flat stale vector shape:", tuple(sample["flat"].shape))
print("context matrix shape:", tuple(sample["context"].shape))
print("valid siblings:", int(sample["mask"].sum().item()))
print("terminal label:", float(sample["label"].item()))

display(pd.DataFrame(sample["context"][sample["mask"]].numpy(), columns=manifest.context_feature_names).head())


## Step 9. Define the Train/Test Split Explicitly

Before training anything, we make the split visible. The ladder uses a grouped out-of-time split ordered by `end_date`, so all rows for the same target market stay on the same side.


In [ ]:
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

config = BeliefUpdatingMVPConfig(
    test_fraction=0.25,
    n_epochs=10,
    batch_size=128,
    encoder_output_dim=32,
    device=device,
)

experiment = BeliefUpdatingMVPExperiment(manifest, config=config)
artifact_bundle = experiment.artifacts(preview_rows=10)

display(artifact_bundle.protocol_summary)
display(artifact_bundle.split_summary)
display(artifact_bundle.train_preview)
display(artifact_bundle.test_preview)


## Step 10. Make Features and Target Explicit

The MVP protocol is a binary classification task. The target is `label` / terminal outcome. The feature blocks depend on the rung:

- rung 1: stale-local features only;
- rung 2: stale-local + aggregated raw context;
- rungs 3-4: stale-local flat vector + variable-length context set.


In [ ]:
target_col = artifact_bundle.protocol_summary.iloc[0]["target_column"]
feature_blocks = artifact_bundle.feature_blocks.copy()

display(feature_blocks)

stale_feature_cols = list(STALE_FEATURE_NAMES)
raw_context_feature_cols = [c for c in AGG_CONTEXT_FEATURE_NAMES if c in train_df.columns] if "train_df" in globals() else [c for c in AGG_CONTEXT_FEATURE_NAMES if c in manifest.examples.columns]

print("target column:", target_col)
print("stale feature columns:", stale_feature_cols)
print("raw context feature columns:", raw_context_feature_cols)
print("context per-sibling feature columns:", list(CONTEXT_FEATURE_NAMES))

display(manifest.examples[stale_feature_cols + [target_col]].head(5))


## Step 11. Make the Models Explicit

The ladder actually uses two model families:

- rungs 1-2: a tabular sklearn pipeline with median imputation, standard scaling, and `HistGradientBoostingClassifier`;
- rungs 3-4: a `DeepSetsEncoder` over sibling context plus a shallow `BeliefUpdatingPredictor` head.


In [ ]:
display(artifact_bundle.model_registry)

gbm_pipeline = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("scale", StandardScaler()),
    ("model", HistGradientBoostingClassifier(
        max_iter=config.gbm_max_iter,
        max_depth=4,
        learning_rate=0.05,
        random_state=config.random_state,
    )),
])

encoder = DeepSetsEncoder(
    input_dim=len(CONTEXT_FEATURE_NAMES),
    hidden_dim=config.encoder_hidden_dim,
    output_dim=config.encoder_output_dim,
)
predictor = BeliefUpdatingPredictor(
    flat_dim=len(STALE_FEATURE_NAMES),
    context_dim=config.encoder_output_dim,
)

print(gbm_pipeline)
print()
print(encoder)
print()
print(predictor)


## Step 12. Make Losses and Metrics Explicit

The task is binary classification, but the training loss depends on the rung:

- rungs 1-2 optimize the built-in gradient-boosting classification objective;
- rungs 3-4 optimize `BCEWithLogitsLoss`, because the PyTorch head returns raw logits.

After training, all rungs are compared with the same evaluation metrics:

- `log_loss`;
- `Brier score`;
- `ROC-AUC`.


In [ ]:
display(artifact_bundle.objective_registry)

criterion = nn.BCEWithLogitsLoss()
print("PyTorch loss for rungs 3-4:", criterion)

train_df, test_df = experiment._split()
example_probs = pd.Series(train_df[target_col].mean(), index=test_df.index, dtype=float).clip(1e-6, 1 - 1e-6)
example_labels = test_df[target_col].to_numpy(dtype=float)

print("example log loss from constant-rate predictor:", round(float(log_loss(example_labels, example_probs)), 5))
print("example Brier score from constant-rate predictor:", round(float(((example_probs - example_labels) ** 2).mean()), 5))
try:
    auc = roc_auc_score(example_labels, example_probs)
except ValueError:
    auc = float("nan")
print("example ROC-AUC from constant-rate predictor:", round(float(auc), 5))


## Step 13. Run the MVP Ladder

Only now do we train. All the ingredients of the experiment have already been made explicit above.


In [ ]:
results = experiment.run(verbose=True)
display(results.to_dataframe())
display(results.compression_summary())


## Step 14. Visualize the MVP Results


In [ ]:
frame = results.to_dataframe()
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

frame.plot(x="rung", y="log_loss", kind="bar", legend=False, ax=axes[0], color="#1D4ED8")
axes[0].set_title("MVP rung log loss")
axes[0].set_xlabel("Rung")
axes[0].set_ylabel("Log loss")
axes[0].tick_params(axis="x", rotation=35)

frame.plot(x="rung", y="roc_auc", kind="bar", legend=False, ax=axes[1], color="#0F766E")
axes[1].set_title("MVP rung ROC-AUC")
axes[1].set_xlabel("Rung")
axes[1].set_ylabel("ROC-AUC")
axes[1].tick_params(axis="x", rotation=35)

plt.tight_layout()


## What to Look For

The MVP claim is supported when:

- `stale_plus_raw` beats `stale_only`;
- `stale_plus_embedding` stays competitive with `stale_plus_raw`;
- `stale_plus_corrupted` falls back toward the stale-only baseline.

That pattern means family context carries real signal and the compact embedding preserves a meaningful fraction of it.
